# Task 2: Streaming Application Documentation

![alt text](file_structure.png)

## 2.0 Configuration

**Source file:** `src/config.py`

**Implementation summary:**
- Defines Kafka producer and topic settings, data file paths, camera speed limits, and streaming parameters.
- Maps each camera source file to a Kafka topic and camera identifier.
- Configures MongoDB connection details for the sink collection.
- Provides the producer batch interval used to simulate real-time emission.

**Relevant code:**
```python
KAFKA_PRODUCER_CONFIG = {
    "bootstrap_servers": KAFKA_BROKERS,
    "client_id": "speed-violation-producer",
    "acks": "all",
    "retries": 3,
    "retry_backoff_ms": 100,
    "compression_type": "gzip",
}

DATA_FILES = {
    "camera_a": {
        "path": "../data/camera_event_A.csv",
        "kafka_topic": "camera-events-camera-1",
        "camera_id": "camera-1",
    },
    "camera_b": {
        "path": "../data/camera_event_B.csv",
        "kafka_topic": "camera-events-camera-2",
        "camera_id": "camera-2",
    },
    "camera_c": {
        "path": "../data/camera_event_C.csv",
        "kafka_topic": "camera-events-camera-3",
        "camera_id": "camera-3",
    },
}

PRODUCER_CONFIG = {
    "batch_interval_seconds": 5,
}
```

**Key justification:** These settings ensure the producer modules know which file to read, which Kafka topic to publish to, and how often to emit each `batch_id` group. The camera speed limits and inter-camera distance metadata are defined centrally so the detection logic stays consistent across the streaming pipeline.

## 2.1.1 Producer Implementation

**Source file:** `src/producer_utils.py`

**Implementation summary:**
- `CameraEventProducer` initializes a Kafka producer with JSON serialization for values and string serialization for keys.
- `load_and_group_events()` reads CSV records using pandas, adds `producer_id`, and groups events by `batch_id`.
- `publish_batch()` sends each event in a batch to Kafka with `ingestion_timestamp`, `batch_id`, and `car_plate` as the message key.
- `stream_events()` publishes batches sequentially and waits `batch_interval_seconds` between batches to emulate streaming behavior.

**Relevant code:**
```python
class CameraEventProducer:
    def __init__(self, config: Dict, producer_id: str) -> None:
        self.config = config
        self.producer_id = producer_id
        self.producer = None
        self.initialize_producer()

    def load_and_group_events(self, csv_path: str) -> Dict[int, List[Dict]]:
        df = pd.read_csv(csv_path)
        df['producer_id'] = self.producer_id
        batches = {
            batch_id: group.to_dict('records') 
            for batch_id, group in df.groupby('batch_id')
        }
        return batches

    def publish_batch(self, topic: str, batch_id: int, events: List[Dict]) -> Tuple[int, int]:
        success_count = 0
        fail_count = 0
        for event in events:
            message_value = {
                **event,
                "ingestion_timestamp": datetime.now(timezone.utc).isoformat(),
                "batch_id": batch_id
            }
            message_key = str(event.get('car_plate', ''))
            self.producer.send(topic, key=message_key, value=message_value)
            success_count += 1
        return success_count, fail_count

    def stream_events(self, csv_path: str, topic: str, batch_interval_seconds: float = 5.0) -> None:
        batches = self.load_and_group_events(csv_path)
        for batch_id in sorted(batches.keys()):
            self.publish_batch(topic, batch_id, batches[batch_id])
            if batch_id < max(batches.keys()):
                time.sleep(batch_interval_seconds)
```

**Relevant requirement coverage:**
- Batches are defined by `batch_id` and published in order.
- Each camera source file is published to its own Kafka topic.
- Producer metadata provides source traceability and supports downstream filtering by producer.

## 2.1.2 Streaming Join Logic and State Management

(in progress)

## 2.1.3 Sink Integration with MongoDB

(in progress)

## 2.1.4 Violation Detection

**Source file:** `src/rules.py`

**Implementation summary:**
- `ViolationDetector.detect_instantaneous()` identifies any event where `speed_reading` exceeds the camera's `speed_limit`.
- `ViolationDetector.detect_average_speed()` calculates average speed between two camera passes and compares it to the ending camera's speed limit.
- `DailyViolationMerger.merge()` consolidates multiple violations for the same `car_plate` and `violation_date` into one aggregated record.

**Relevant code:**
```python
class ViolationDetector:
    def detect_instantaneous(self, enriched_df: DataFrame) -> DataFrame:
        return enriched_df.filter(F.col('speed_reading') > F.col('speed_limit')) \
            .withColumn('violation_type', F.lit(ViolationType.SPEED.value)) \
            .withColumn('violation_date', F.date_format(F.col('timestamp'), 'yyyy-MM-dd')) \
            .withColumn('event_details', F.struct(
                F.col('camera_id'),
                F.col('timestamp'),
                F.col('speed_reading').alias('speed')
            ))

    def detect_average_speed(self, joined_df: DataFrame) -> DataFrame:
        time_diff_hours = (F.unix_timestamp('timestamp_end') - F.unix_timestamp('timestamp_start')) / 3600.0
        distance = F.abs(F.col('camera_id_end') - F.col('camera_id_start'))
        result_df = joined_df.withColumn('avg_speed', distance / time_diff_hours) \
            .filter(F.col('avg_speed') > F.col('speed_limit_end')) \
            .withColumn('violation_type', F.lit(ViolationType.AVERAGE_SPEED.value)) \
            .withColumn('violation_date', F.date_format(F.col('timestamp_end'), 'yyyy-MM-dd'))

        return result_df.withColumn('event_details', F.struct(
                F.col('camera_id_start'),
                F.col('camera_id_end'),
                F.col('timestamp_start'),
                F.col('timestamp_end'),
                F.col('avg_speed')
            )) \
            .select(
                F.col('start.car_plate').alias('car_plate'),
                'violation_date',
                'violation_type',
                F.col('speed_limit_end').alias('speed_limit'),
                'event_details'
            )

class DailyViolationMerger:
    def merge(self, violations_df: DataFrame) -> DataFrame:
        return violations_df.groupBy('car_plate', 'violation_date') \
            .agg(
                F.collect_list(
                    F.struct(
                        'violation_type', 
                        'speed_limit', 
                        'event_details'
                    )
                ).alias('violations')
            )
```

**Relevant requirement coverage:**
- Instantaneous camera violations are detected when a single reading exceeds the localized limit.
- Segment average-speed violations are computed using time difference and camera positions.
- Multiple violations on the same date are merged into one aggregated daily record per vehicle.